# Common Skin Disease Classification — Vision Transformer (ViT-B/16) GPU Training
### Google Colab Ready-to-Run Training Notebook

**Project**: AI-Based Skin Disease Detection with Treatment Cost Estimation  
**Model Architecture**: Vision Transformer Base (`vit_base_patch16_224` via `timm`)  
**Target Categories (9 Classes)**:
1. Acne (`acne`)
2. Allergic Contact Dermatitis (`allergic_contact_dermatitis`)
3. Eczema (`eczema`)
4. Impetigo (`impetigo`)
5. Psoriasis (`psoriasis`)
6. Ringworm / Tinea Corporis (`ringworm`)
7. Scabies (`scabies`)
8. Seborrheic Dermatitis (`seborrheic_dermatitis`)
9. Urticaria / Hives (`urticaria`)

**Excluded Categories (Documented Data Insufficiency)**:
- *Vitiligo*: Insufficient verified clinical photographs (<10 samples across Google SCIN and SkinDisNet).
- *Other fungal skin infections*: Insufficient samples (only 28 Tinea Versicolor cases); excluded to preserve clinical separation from verified Tinea Corporis.

## 1. GPU Verification

In [ ]:
!nvidia-smi

import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('Device Capability:', torch.cuda.get_device_capability(0))
else:
    print('WARNING: Running on CPU. Please switch Colab Runtime to T4 or V100 GPU.')

## 2. Dependency Installation

In [ ]:
!pip install -q timm torch torchvision scikit-learn matplotlib seaborn pandas pillow

## 3. Dataset Extraction

In [ ]:
import os
from pathlib import Path

# If package zip is present in Colab workspace
if os.path.exists('colab_common_skin_package.zip'):
    !unzip -q colab_common_skin_package.zip
    print('Extracted colab_common_skin_package.zip successfully.')
else:
    print('colab_common_skin_package.zip not found. Please upload it to Colab files pane.')

# Verify directory structure
print('Images count:', len(list(Path('data/common_skin/images').glob('*'))))
print('Manifests found:', [f.name for f in Path('data/manifests').glob('common_skin*.csv')])

## 4. Dataset and DataLoader Verification

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from training.common_skin_dataset import build_common_skin_dataloaders

train_loader, val_loader, test_loader, stats = build_common_skin_dataloaders(batch_size=16)
print('DataLoader Summary:', stats)

# Display first batch sample
images, labels, metas = next(iter(train_loader))
print('Batch images tensor shape:', images.shape)
print('Batch labels:', labels[:8].tolist())

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
for i in range(4):
    img = images[i].permute(1, 2, 0).numpy() * std + mean
    img = img.clip(0, 1)
    axes[i].imshow(img)
    axes[i].set_title(f"{metas['target_class'][i]}\n({metas['skin_tone'][i]})")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 5. Model Training (ViT-B/16 with AdamW & CosineAnnealingLR)

In [ ]:
from training.train_common_skin import train_common_skin_model

# Run training for up to 25 epochs with mixed precision
evaluation_results = train_common_skin_model(
    epochs=25,
    batch_size=16,
    lr=1e-4,
    weight_decay=0.01,
    patience=5,
    unfreeze_blocks=2,
    smoke_test=False
)

## 6. Training Curve Visualization

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

curves_path = 'training/results/common_skin_plots/common_skin_training_curves.png'
if os.path.exists(curves_path):
    img = Image.open(curves_path)
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 7. Confusion Matrix and Test Predictions

In [ ]:
cm_path = 'training/results/common_skin_plots/common_skin_confusion_matrix.png'
if os.path.exists(cm_path):
    img = Image.open(cm_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 8. Final Evaluation Metrics

In [ ]:
import json
import pandas as pd

with open('training/results/common_skin_metrics.json') as f:
    metrics = json.load(f)

print('Overall Accuracy:     ', f"{metrics['overall_accuracy']:.4f}")
print('Balanced Accuracy:    ', f"{metrics['balanced_accuracy']:.4f}")
print('Macro Precision:      ', f"{metrics['macro_precision']:.4f}")
print('Macro Recall:         ', f"{metrics['macro_recall']:.4f}")
print('Macro F1-Score:       ', f"{metrics['macro_f1']:.4f}")
print('Weighted F1-Score:    ', f"{metrics['weighted_f1']:.4f}")
print('Multiclass ROC-AUC:   ', metrics.get('roc_auc_macro'))

print('\n--- Per-Class Metrics ---')
per_class_df = pd.DataFrame(metrics['per_class_metrics']).T
print(per_class_df.to_string())

if metrics.get('skin_tone_breakdown'):
    print('\n--- Performance Across Skin Tones ---')
    st_df = pd.DataFrame(metrics['skin_tone_breakdown']).T
    print(st_df.to_string())

## 9. Model Checkpoint & Results Download

In [ ]:
import zipfile
from google.colab import files

# Archive results
with zipfile.ZipFile('common_skin_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk('training/results'):
        for fn in fnames:
            if 'common_skin' in fn or 'common_skin' in root:
                fpath = os.path.join(root, fn)
                z.write(fpath, os.path.relpath(fpath, 'training/results'))

print('Downloading models/common_skin_model.pth...')
files.download('models/common_skin_model.pth')

print('Downloading common_skin_results.zip...')
files.download('common_skin_results.zip')